# Fine-tune detector de cabezas — dataset filtrado (≥5 cabezas) · **ejecución NATIVA**

Versión para correr **dentro del contenedor `mot-dev`** (JupyterLab con GPU + ultralytics + torch CUDA).
Aquí el entrenamiento corre nativo con `from ultralytics import YOLO` — NO usa `docker run`.

> Si en cambio abres Jupyter en el **host**, usa `train_head_min5_filtered.ipynb` (ese sí lanza Docker).

**Hipótesis:** ~65% de los frames de `s03` (70% del dataset) están mal etiquetados (mediana **1**
cabeza en multitudes) → el modelo aprende "cabeza = fondo" y subcuenta (golden bias -1.87, recall 0.79).
Construimos `bus_head_v5_min5` = frames con **≥5 cabezas** y hacemos fine-tune **desde el base**.

**Campeón R5 a batir:** golden count-MAE **2.04** · recall **0.792** · mAP50 **0.872**.

## 0. Configuración (rutas dentro del contenedor: `/workspace`)

In [1]:
import os, shutil, hashlib, re
from pathlib import Path
os.environ.setdefault('YOLO_CONFIG_DIR', '/tmp/Ultralytics')  # /root/.config no es escribible

REPO       = Path('/workspace')                 # repo montado en el contenedor
SRC        = REPO / 'data' / 'bus_head_v5'
DST_NAME   = 'bus_head_v5_min5'
DST        = REPO / 'data' / DST_NAME
YAML_PATH  = REPO / 'data' / f'{DST_NAME}.yaml'
MIN_HEADS  = 5

BASE_MODEL = REPO / 'models' / 'yolov5mu-head-base.pt'
PROJECT    = str(REPO / 'outputs' / 'head_detector')
RUN_NAME   = 'yolo-bus-head-min5'
EPOCHS, IMGSZ, BATCH, WORKERS = 80, 640, 8, 4

assert SRC.exists(),        f'no existe {SRC} — ¿montaste el repo en /workspace?'
assert BASE_MODEL.exists(), f'no existe {BASE_MODEL}'
print('REPO :', REPO)
print('base :', BASE_MODEL)
print('umbral:', MIN_HEADS, 'cabezas/frame')

REPO : /workspace
base : /workspace/models/yolov5mu-head-base.pt
umbral: 5 cabezas/frame


## 1. Construir el dataset filtrado (≥5 cabezas/frame)

In [2]:
def n_boxes(p: Path) -> int:
    if not p.exists():
        return 0
    with open(p) as f:
        return sum(1 for ln in f if ln.strip())

def source_of(stem: str) -> str:
    s = re.sub(r'_f?\d+$', '', stem)
    return re.sub(r'\d+$', '', s) or stem

if DST.exists():
    shutil.rmtree(DST)

stats, by_source, total_boxes = {}, {}, 0
for split in ('train', 'val'):
    out_img = DST / 'images' / split; out_img.mkdir(parents=True, exist_ok=True)
    out_lbl = DST / 'labels' / split; out_lbl.mkdir(parents=True, exist_ok=True)
    kept = dropped = 0
    for img in sorted((SRC / 'images' / split).iterdir()):
        if img.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
            continue
        lbl = SRC / 'labels' / split / (img.stem + '.txt')
        nb = n_boxes(lbl)
        if nb >= MIN_HEADS:
            shutil.copy2(img, out_img / img.name)
            shutil.copy2(lbl, out_lbl / lbl.name)
            kept += 1; total_boxes += nb
            src = source_of(img.stem); by_source[src] = by_source.get(src, 0) + 1
        else:
            dropped += 1
    stats[split] = (kept, dropped)
    print(f'[{split}] copiados={kept}  descartados={dropped}')

tk = sum(k for k, _ in stats.values()); ta = sum(k + d for k, d in stats.values())
print(f'\nTOTAL: {ta} -> {tk} frames  ({ta-tk} descartados, {100*(ta-tk)/ta:.0f}%)  cajas={total_boxes}')
print('\nfuentes conservadas:')
for s, n in sorted(by_source.items(), key=lambda x: -x[1]):
    print(f'  {s:12s} {n}')

[train] copiados=1044  descartados=1028
[val] copiados=63  descartados=305

TOTAL: 2440 -> 1107 frames  (1333 descartados, 55%)  cajas=13146

fuentes conservadas:
  s            602
  v            207
  videoTM_     84
  v04_         78
  v16b         42
  v05_         36
  v18S         36
  v10b         22


## 2. Anti-fuga del golden

In [3]:
GOLDEN = REPO / 'data' / 'golden' / 'images' / 'val'
def cam_frame(stem):
    m = re.search(r'(\d+)$', stem.replace('_f', '_'))
    return (source_of(stem), m.group(1) if m else '')

gids = {cam_frame(p.stem) for p in GOLDEN.iterdir()
        if p.suffix.lower() in ('.jpg', '.jpeg', '.png')} if GOLDEN.exists() else set()
tids = {cam_frame(p.stem) for split in ('train', 'val')
        for p in (DST / 'images' / split).iterdir()}
leak = gids & tids
print(f'golden={len(gids)}  dataset={len(tids)}')
if leak:
    raise SystemExit(f'⚠️ FUGA del golden: {sorted(leak)[:10]}')
print('✅ sin fuga del golden')

golden=151  dataset=1096
✅ sin fuga del golden


## 3. Escribir el YAML

In [4]:
YAML_PATH.write_text(
    f'path: /workspace/data/{DST_NAME}\ntrain: images/train\nval: images/val\n\nnames:\n  0: head\n'
)
print('escrito:', YAML_PATH)
print(YAML_PATH.read_text())

escrito: /workspace/data/bus_head_v5_min5.yaml
path: /workspace/data/bus_head_v5_min5
train: images/train
val: images/val

names:
  0: head



## 4. Pre-check de GPU (nativo)

In [5]:
import torch
print('torch', torch.__version__, '| cuda disponible:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'sin CUDA — lanza el contenedor con --runtime=nvidia -e NVIDIA_VISIBLE_DEVICES=all'
print('GPU:', torch.cuda.get_device_name(0))

torch 2.2.2 | cuda disponible: True
GPU: NVIDIA GeForce RTX 4060 Ti


## 5. Entrenar (fine-tune desde el base, ~1–2 h)

Corre en el kernel y va mostrando el progreso por época. ⚠️ Si el kernel se cae, el entreno se detiene.
El contenedor se lanzó con `--shm-size=8g`, necesario para los workers del DataLoader.

In [6]:
from ultralytics import YOLO
model = YOLO(str(BASE_MODEL))
results = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, workers=WORKERS, cache=False, device=0,
    project=PROJECT, name=RUN_NAME,
)
print('\n✅ entrenamiento terminado')

WARNING ⚠️ user config directory '/tmp/Ultralytics/Ultralytics' is not writable, using '/tmp/Ultralytics'. Set YOLO_CONFIG_DIR to override.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/tmp/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
WARNING ⚠️ /workspace/models/yolov5mu-head-base.pt appears to require 'dill', which is not in Ultralytics requirements.
AutoInstall will run now for 'dill' but this feature will be removed in the future.
Recommend fixes are to train a new model using the latest 'ultralytics' package or to run a command with an official Ultralytics model, i.e. 'yolo predict model=yolo11n.pt'
requirements: Ultralytics requirement ['dill'] not found, attempting AutoUpdate...

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install

2026/06/18 00:25:27 INFO mlflow.tracking.fluent: Experiment with name '/workspace/outputs/head_detector' does not exist. Creating a new experiment.


MLflow: logging run_id(eb139e5ada9d494b8b6fbd9a63b271cc) to runs/mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs/mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to /workspace/outputs/head_detector/yolo-bus-head-min5
Starting training for 80 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/80      3.43G      2.048      1.477      1.656         85        640: 100% ━━━━━━━━━━━━ 131/131 4.3it/s 30.3s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 3.0it/s 1.3s0.6ss
                   all         63        475      0.656      0.533      0.582      0.231

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/80      3.65G      1.728     0.9664       1.45         74        640: 100% ━━━━━━━━━━━━ 131/131 5.6it/s 23.3s0.2s


## 6. Localizar `best.pt` + sha256

In [7]:
best = Path(PROJECT) / RUN_NAME / 'weights' / 'best.pt'
if not best.exists():
    cands = [p for p in REPO.rglob('best.pt') if RUN_NAME in str(p)]
    best = max(cands, key=lambda p: p.stat().st_mtime) if cands else best
if best.exists():
    print('best.pt:', best)
    print('sha256 :', hashlib.sha256(best.read_bytes()).hexdigest())
else:
    print('⚠️ best.pt no encontrado')

best.pt: /workspace/outputs/head_detector/yolo-bus-head-min5/weights/best.pt
sha256 : d158852aad4218c85430893c31cd74456bca907294c253c5aa32648c97186e85


## 7. Evaluar contra el golden (recall / mAP)

Para el veredicto oficial **count-MAE vs R5** usa el skill `/eval-golden` desde Claude Code.

In [8]:
m = YOLO(str(best))
metrics = m.val(data=str(REPO / 'data' / 'golden' / 'golden.yaml'),
                imgsz=IMGSZ, conf=0.25, iou=0.5,
                project=PROJECT, name=f'{RUN_NAME}-golden')
print('\n=== golden (vs R5: recall 0.792 | mAP50 0.872) ===')
print(f'precision : {metrics.box.mp:.3f}')
print(f'recall    : {metrics.box.mr:.3f}')
print(f'mAP50     : {metrics.box.map50:.3f}')
print(f'mAP50-95  : {metrics.box.map:.3f}')
print('\ncount-MAE oficial -> ejecuta  /eval-golden  en Claude Code')

Ultralytics 8.4.0 🚀 Python-3.10.14 torch-2.2.2 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 15927MiB)
YOLOv5m summary (fused): 107 layers, 25,045,795 parameters, 0 gradients, 64.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 181.3±27.0 MB/s, size: 128.8 KB)
val: Scanning /workspace/data/golden/labels/val.cache... 151 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 151/151 42.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 6.2it/s 1.6s.2s
                   all        151       1241      0.877      0.809      0.873      0.634
Speed: 1.0ms preprocess, 7.9ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /workspace/outputs/head_detector/yolo-bus-head-min5-golden

=== golden (vs R5: recall 0.792 | mAP50 0.872) ===
precision : 0.877
recall    : 0.809
mAP50     : 0.873
mAP50-95  : 0.634

count-MAE oficial -> ejecuta  /eval-golden  en Claude Code
